# Notebook 1: Modell-Inspektion
**Kurs:** Mechanistic Interpretability  
**Modell:** EleutherAI/pythia-410m  
**Ziel:** Pythia-410M laden, Architektur verstehen und erste Vorhersagen generieren.

> **Aufgabe:** Implementiere die Zellen schrittweise. Hilfreiche Dokumentation:
> - HuggingFace Transformers: https://huggingface.co/docs/transformers/
> - Pythia-Modell: https://huggingface.co/EleutherAI/pythia-410m
> - PyTorch: https://pytorch.org/docs/stable/

## 1. Installation & Imports

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import torch
import os
# TODO: Installiere benötigte Pakete (auskommentieren und einmalig ausführen)
# \!pip install transformers torch accelerate

# TODO: Importiere die folgenden Bibliotheken:
#   - torch
#   - numpy as np
#   - matplotlib.pyplot as plt
#   - AutoModelForCausalLM, AutoTokenizer aus transformers

# Hinweis: torch.cuda.is_available() prüft, ob eine GPU vorhanden ist
# Hinweis: Dokumentation zu AutoModelForCausalLM:
#   https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForCausalLM

## 2. Modell und Tokenizer laden

In [5]:
MODEL_NAME = "EleutherAI/pythia-410m"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Modell laden
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",   # wichtig für Attention-Ausgabe
    torch_dtype=torch.float32
)

# Auf Gerät verschieben
model = model.to(device)

# Eval-Modus
model.eval()

print("Modell geladen!")
print("Device:", device)
# Fallback, falls Pythia nicht lädt: MODEL_NAME = "gpt2-medium"

# TODO: Lade Tokenizer und Modell von HuggingFace
#   - AutoTokenizer.from_pretrained(MODEL_NAME)
#   - AutoModelForCausalLM.from_pretrained(MODEL_NAME)
#   - Verschiebe das Modell auf das richtige Gerät (.to(device))
#   - Setze das Modell in den Evaluierungs-Modus (.eval())
#   - Setze das Attn-Implementation auf "eager". Achtung, wenn dies nicht geladen wird, dann werden Attentions-Schichten nicht korrekt zurückgegeben.
# model = # AutoModelForCausalLM.from_pretrained(                                                                                       #     
#      MODEL_NAME, 
#      attn_implementation="eager",   # <-- entscheidend
#      #torch_dtype=torch.float32,                                                                                                          
#  ).to(device)
#  model.eval()  


# Hinweis: Falls der Tokenizer keinen pad_token hat, setze:
#   tokenizer.pad_token = tokenizer.eos_token
#
# Dokumentation: https://huggingface.co/docs/transformers/main_classes/tokenizer

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Modell geladen!
Device: cuda


In [6]:


texts = [
    "tina is a girl.she has cat",
    "Berlin is the capital of",
    "AI",                    
]
tokens = tokenizer(texts,  padding=True )

print(tokens)

{'input_ids': [[85, 1758, 310, 247, 3226, 15, 6689, 556, 5798], [23666, 3642, 310, 253, 5347, 273, 1, 1, 1], [18128, 1, 1, 1, 1, 1, 1, 1, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0]]}


## 3. Architekturübersicht

In [7]:
# TODO: Gib die wichtigsten Architektur-Parameter aus:
#   - model.config.model_type
#   - model.config.hidden_size
#   - model.config.num_hidden_layers
#   - model.config.num_attention_heads
#   - model.config.vocab_size
#   - model.config.max_position_embeddings
#
# Hinweis: head_dimension = hidden_size / num_attention_heads
# Hinweis: Alle Konfigurationsparameter: https://huggingface.co/docs/transformers/model_doc/gpt_neox#transformers.GPTNeoXConfig


print("Model type:", model.config.model_type)
print("Hidden size:", model.config.hidden_size)
print("Hidden layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("Vocabulary size:", model.config.vocab_size)
print("Max position embeddings:", model.config.max_position_embeddings)

# محاسبه ابعاد هر Head
head_dimension = (
    model.config.hidden_size /
    model.config.num_attention_heads
)

print("Head dimension:", head_dimension)

Model type: gpt_neox
Hidden size: 1024
Hidden layers: 24
Attention heads: 16
Vocabulary size: 50304
Max position embeddings: 2048
Head dimension: 64.0


In [49]:
model

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 1024)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-23): 24 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=1024, out_features=3072, bias=True)
          (dense): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=1024, out_features=4096, bias=True)
          (dense_4h_to_h): Linear(in_features=4096, out_features=1024, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((1024,), eps=1e-05, 

## 4. Parameter-Zählung

In [8]:
# TODO: Schreibe eine Funktion count_params(module) die die Gesamtzahl
#   der Parameter in einem Modul zählt.
#   Tipp: module.parameters() liefert alle Parameter-Tensoren.
#   Tipp: p.numel() gibt die Anzahl der Elemente eines Tensors zurück.
#
# TODO: Berechne die Parameterzahl für:
#   - Token-Embedding: model.gpt_neox.embed_in
#   - Jede Transformer-Schicht: model.gpt_neox.layers[i]
#   - Final LayerNorm: model.gpt_neox.final_layer_norm
#   - Unembed (LM-Head): model.embed_out
#
# Tipp: Prozentualer Anteil = component_params / total_params * 100

def count_params(module):
    return sum(p.numel() for p in module.parameters())

total_params = count_params(model)

print(f"Total params: {total_params:,}\n")

# Token Embedding
embed_params = count_params(model.gpt_neox.embed_in)

print(
    f"Embedding: {embed_params:,} "
    f"({embed_params/total_params*100:.2f}%)"
)

    

Total params: 405,334,016

Embedding: 51,511,296 (12.71%)


## 5. Top-K Vorhersagen

In [9]:
# TODO: Schreibe eine Funktion top_k_predictions(prompt, k=5) die:
#   1. Den Prompt tokenisiert (tokenizer(prompt, return_tensors="pt"))
#   2. Einen Forward Pass durchführt (model(**inputs))
#   3. Die Logits des LETZTEN Tokens nimmt (outputs.logits[0, -1, :])
#   4. Softmax anwendet und die Top-K Tokens zurückgibt
#
# Hinweis: torch.topk(probs, k) gibt Werte und Indizes zurück
# Hinweis: tokenizer.decode([token_id]) konvertiert einen Token-Index in Text
#
# Dokumentation zu torch.topk:
#   https://pytorch.org/docs/stable/generated/torch.topk.html
#
# Teste deine Funktion mit den Prompts:
#   "The capital of France is"
#   "The Eiffel Tower is located in"
#   "Python is a programming"
def top_k_predictions(prompt, k=5):

    # 1) Prompt tokenisieren
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # auf CPU/GPU verschieben
    inputs = {key: value.to(device)
              for key, value in inputs.items()}

    # Kein Gradient nötig
    with torch.no_grad():
        outputs = model(**inputs)

    # 3) Logits des letzten Tokens
    logits = outputs.logits[0, -1, :]

    # 4) Softmax → Wahrscheinlichkeiten
    probs = torch.softmax(logits, dim=-1)

    # Top-K
    top_probs, top_ids = torch.topk(probs, k)

    results = []

    for prob, token_id in zip(top_probs, top_ids):

        token_text = tokenizer.decode([token_id.item()])

        results.append(
            (token_text, prob.item())
        )

    return results

#TODO 
prompts = [
    "the movie was realy good. The sentiment is",
    "The service was excellent. The sentiment is",
    "The review says the product is broken. It is"
]
for p in prompts:

    print("\nPrompt:", p)

    preds = top_k_predictions(p,10)

    for token, prob in preds:
        print(f"{token:<15} {prob:.4f}")


Prompt: the movie was realy good. The sentiment is
 real           0.0501
 good           0.0492
 great          0.0469
 very           0.0404
 really         0.0278
 a              0.0271
 true           0.0267
 right          0.0250
 not            0.0246
 there          0.0234

Prompt: The service was excellent. The sentiment is
 that           0.0761
 good           0.0389
 the            0.0341
 very           0.0327
 great          0.0321
 not            0.0303
 a              0.0277
 one            0.0184
 positive       0.0154
 mutual         0.0146

Prompt: The review says the product is broken. It is
 not            0.1154
 a              0.1141
 also           0.0409
 the            0.0243
 an             0.0180
 possible       0.0132
 unclear        0.0132
 designed       0.0122
 broken         0.0114
 in             0.0105


## 6. Clean vs. Corrupted Vergleich

In [10]:
clean_prompt = "The movie was amazing. The sentiment is"
corrupted_prompt = "The movie was terrible. The sentiment is"

# TODO: Vergleiche die Top-5-Vorhersagen für beide Prompts nebeneinander.
#   Verwende deine top_k_predictions-Funktion aus Aufgabe 5.
#
# Frage zur Reflexion: Wie unterscheiden sich die Vorhersagen?
# Was sagt das über das "Weltmodell" des Transformers?

clean_preds = top_k_predictions(clean_prompt, k=10)
corrupt_preds = top_k_predictions(corrupted_prompt, k=10)

print(f"{'Clean Prompt':<25} | {'Corrupted Prompt':<25}")
print("-" * 55)

for (clean_token, clean_prob), (corr_token, corr_prob) in zip(clean_preds, corrupt_preds):
    print(
        f"{clean_token:<15} {clean_prob:.4f} | "
        f"{corr_token:<15} {corr_prob:.4f}"
    )

Clean Prompt              | Corrupted Prompt         
-------------------------------------------------------
 great          0.0632 |  not            0.0471
 so             0.0457 |  that           0.0340
 very           0.0426 |  a              0.0322
 really         0.0326 |  so             0.0220
 true           0.0268 |  very           0.0217
 the            0.0237 |  just           0.0216
 right          0.0230 |  the            0.0207
 what           0.0229 |  pretty         0.0185
 a              0.0225 |  really         0.0178
 that           0.0205 |  all            0.0162


## 7. Token-Embeddings visualisieren

In [48]:
# TODO: Greife auf die Token-Embedding-Matrix zu:
#   model.gpt_neox.embed_in.weight  → Shape: (vocab_size, hidden_size)
#
# TODO: Wähle 10 interessante Token (z.B. " Paris", " London", " cat", ...)
#   und berechne ihre L2-Norm (vec.norm().item())
#   Tipp: tokenizer.encode(" Paris") gibt die Token-ID zurück
#
# TODO: Erstelle ein Balkendiagramm der L2-Normen (plt.barh)
#
# BONUS: Berechne die Cosine-Ähnlichkeit zwischen " Paris" und allen anderen
#   Vokabular-Token und zeige die Top-10 ähnlichsten Token.
#   Tipp: torch.nn.functional.cosine_similarity(a, b)
#   Dokumentation: https://pytorch.org/docs/stable/nn.functional.html
emb = model.gpt_neox.embed_in.weight
print("Embedding shape:", emb.shape)

tokens = [
    " Paris",
    " London",
    " Berlin",
    " movie",
    " good",
    " bad",
    " food",
    " positive",
    " negative",
    " Python"
]

norms = []

for tok in tokens:
    token_ids = tokenizer.encode(tok)
    token_id = token_ids[0]

    vec = emb[token_id]
   # print(vec)
    norm = vec.norm().item()
    print(tok, "→ ID:", token_id, "Norm:", norm)


Embedding shape: torch.Size([50304, 1024])
 Paris → ID: 7785 Norm: 0.8038759827613831
 London → ID: 4693 Norm: 0.7978219985961914
 Berlin → ID: 12911 Norm: 0.8061519265174866
 movie → ID: 6440 Norm: 0.7975524067878723
 good → ID: 1175 Norm: 0.7807690501213074
 bad → ID: 3076 Norm: 0.8059031963348389
 food → ID: 2739 Norm: 0.8024579286575317
 positive → ID: 2762 Norm: 0.8082696795463562
 negative → ID: 4016 Norm: 0.8084568977355957
 Python → ID: 13814 Norm: 0.8081262111663818


In [44]:
t=torch.tensor([1,2] , dtype=torch.float32)

t.norm().item()

2.2360680103302

## 8. Zusammenfassung

In [18]:
# TODO: Beantworte folgende Fragen als print()-Ausgaben oder Kommentare:
#
# 1. Wie viele Parameter hat Pythia-410M insgesamt?
# 2. Welche Komponente hat den größten Anteil an den Parametern?
# 3. Was ist die Dimension jedes Attention Heads?
# 4. Wie unterscheiden sich die Vorhersagen für den clean und den corrupted Prompt?
# 5. Welche Token sind im Embedding-Raum " Paris" am ähnlichsten?

print("Meine Erkenntnisse:")
# Deine Antworten hier...

Meine Erkenntnisse:
